# Level 15 — Labelled Factor Regression

**Audience:** analysts evaluating fund or strategy return exposures.

**Prerequisites:** basic linear regression, periodic decimal returns, and
chronological train/evaluation discipline.

**Learning goals**

1. recover labelled alpha and factor betas from synthetic returns;
2. interpret fit and classical inference diagnostics;
3. inspect rolling exposure changes;
4. create additive model-implied factor attribution.

No Fama–French dataset is bundled. Users remain responsible for factor
definitions, licensing, frequency, and risk-free-rate conventions.

## 1. Setup and synthetic factors

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.analytics import (
    factor_regression,
    factor_return_attribution,
    rolling_factor_regression,
)

In [ ]:
rng = np.random.default_rng(20260728)
dates = pd.date_range("2018-01-31", periods=96, freq="ME")
factors = pd.DataFrame(
    {
        "Market": rng.normal(0.006, 0.040, len(dates)),
        "Value": rng.normal(0.002, 0.025, len(dates)),
        "Quality": rng.normal(0.002, 0.020, len(dates)),
    },
    index=dates,
)
noise = rng.normal(0.0, 0.004, len(dates))
fund_returns = pd.Series(
    0.001
    + 1.10 * factors["Market"]
    + 0.35 * factors["Value"]
    - 0.20 * factors["Quality"]
    + noise,
    index=dates,
    name="Synthetic fund",
)
pd.concat([fund_returns, factors], axis=1).head()

## 2. Static OLS exposures

The regression models fund excess return as an intercept plus labelled factor
returns. Alpha is annualized by multiplication; betas remain unitless.

In [ ]:
result = factor_regression(
    fund_returns,
    factors,
    periods_per_year=12,
)
pd.DataFrame(
    {
        "coefficient": result.coefficients,
        "standard_error": result.standard_errors,
        "t_statistic": result.t_statistics,
        "p_value": result.p_values,
    }
)

In [ ]:
pd.Series(
    {
        "r_squared": result.r_squared,
        "adjusted_r_squared": result.adjusted_r_squared,
        "annualized_residual_volatility": result.residual_volatility,
        "complete_observations": result.n_observations,
    }
)

## 3. Rolling exposures and a regime change

Create a second synthetic fund whose Market and Value betas change halfway
through the sample. Trailing windows reveal the transition gradually.

In [ ]:
changing_fund = pd.Series(
    np.concatenate(
        [
            (
                0.001
                + 1.30 * factors["Market"].iloc[:48]
                + 0.10 * factors["Value"].iloc[:48]
                + noise[:48]
            ).to_numpy(),
            (
                -0.0005
                + 0.55 * factors["Market"].iloc[48:]
                + 0.85 * factors["Value"].iloc[48:]
                + noise[48:]
            ).to_numpy(),
        ]
    ),
    index=dates,
    name="Changing fund",
)
rolling = rolling_factor_regression(
    changing_fund,
    factors,
    window=36,
    step=6,
    periods_per_year=12,
)
rolling.betas.tail()

## 4. Additive model-implied attribution

Attribution multiplies each periodic factor return by its fitted exposure and
adds periodic alpha. Realized fund return can still differ by the residual.

In [ ]:
attribution = factor_return_attribution(
    result.betas,
    factors,
    alpha=result.alpha / 12,
)
attribution.head()

In [ ]:
pd.DataFrame(
    {
        "model_fitted": attribution["total"],
        "regression_fitted": result.fitted_returns,
        "realized_fund": fund_returns,
        "residual": result.residuals,
    }
).head()

## Exercise — compare stable and changing exposures

Run the same 36-month rolling regression on the stable synthetic fund. Compare
the standard deviation of rolling betas with the changing fund.

In [ ]:
# Try it here.
stable_rolling = rolling_factor_regression(
    fund_returns,
    factors,
    window=36,
    step=6,
    periods_per_year=12,
)

### Answer scaffold

In [ ]:
pd.DataFrame(
    {
        "stable_beta_std": stable_rolling.betas.std(),
        "changing_beta_std": rolling.betas.std(),
        "stable_last_beta": stable_rolling.betas.iloc[-1],
        "changing_last_beta": rolling.betas.iloc[-1],
    }
)

## Interpretation and pitfalls

- Factor exposure is descriptive association, not proof of causality or skill.
- Factor returns and fund returns must use consistent frequency, currency, and
  return conventions.
- Alpha depends on factor choice and the risk-free-rate convention.
- Classical standard errors here assume homoskedastic residuals; HAC inference
  is not claimed.
- Rolling windows are trailing and descriptive, not forecasts.
- Collinear factors make unconstrained OLS exposures unidentified.
- High R-squared does not establish out-of-sample persistence.

Useful extensions include chronological holdout review, robust inference, and
regularized models for wider or collinear factor sets.